In [1]:
import pandas as pd

df=pd.read_csv("data/data_date.csv")
df.head()

,Date,Country,Status,AQI Value
0,2022-07-21,Albania,Good,14
1,2022-07-21,Algeria,Moderate,65
2,2022-07-21,Andorra,Moderate,55
3,2022-07-21,Angola,Unhealthy for Sensitive Groups,113
4,2022-07-21,Argentina,Moderate,63


In [2]:
import pandas as pd

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Country', 'Date'])

In [3]:
df = df.drop(columns=['Status'])

In [4]:
df['Country_ID'] = df['Country'].astype('category').cat.codes
df = df.drop(columns=['Country'])

In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['AQI Value'] = scaler.fit_transform(df[['AQI Value']])

In [6]:
import numpy as np

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

In [7]:
values = df.groupby('Country_ID')['AQI Value'].apply(list)
values

Country_ID
0      [0.013513513513513514, 0.016632016632016633, 0...
1      [0.06652806652806653, 0.06652806652806653, 0.0...
2      [0.056133056133056136, 0.05197505197505198, 0....
3      [0.11642411642411643, 0.15696465696465697, 0.1...
4      [0.06444906444906445, 0.062370062370062374, 0....
                             ...                        
137    [0.061330561330561334, 0.059251559251559255, 0...
138    [0.07172557172557173, 0.07172557172557173, 0.0...
139    [0.002079002079002079, 0.0010395010395010396, ...
140    [0.04365904365904366, 0.04158004158004158, 0.0...
141    [0.0395010395010395, 0.0550935550935551, 0.036...
Name: AQI Value, Length: 142, dtype: object

In [8]:
SEQ_LEN = 7
X_all, y_all = [], []

for _, group in df.groupby('Country_ID'):
    values = group[['AQI Value']].values
    if len(values) > SEQ_LEN:
        X, y = create_sequences(values, SEQ_LEN)
        X_all.append(X)
        y_all.append(y)

X = np.concatenate(X_all)
y = np.concatenate(y_all)

In [9]:
X.shape, y.shape

((22657, 7, 1), (22657, 1))

In [10]:
X[0:2],y[0:2]

(array([[[0.01351351],
         [0.01663202],
         [0.01455301],
         [0.01455301],
         [0.01975052],
         [0.01455301],
         [0.01975052]],
 
        [[0.01663202],
         [0.01455301],
         [0.01455301],
         [0.01975052],
         [0.01455301],
         [0.01975052],
         [0.01767152]]]),
 array([[0.01767152],
        [0.05613306]]))

In [11]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [12]:
import torch
import torch.nn as nn

class AQILSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

In [13]:
model = AQILSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
from tqdm import tqdm

EPOCHS = 40

for epoch in tqdm(range(EPOCHS)):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(torch.tensor(X_train, dtype=torch.float32))
    loss = criterion(outputs, torch.tensor(y_train, dtype=torch.float32))
    
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

  2%|▎         | 1/40 [00:02<01:20,  2.08s/it]

Epoch 0, Loss: 0.0030


 15%|█▌        | 6/40 [00:11<01:05,  1.93s/it]

Epoch 5, Loss: 0.0029


 28%|██▊       | 11/40 [00:18<00:43,  1.52s/it]

...

 78%|███████▊  | 31/40 [00:41<00:08,  1.02it/s]

Epoch 30, Loss: 0.0021


 90%|█████████ | 36/40 [00:47<00:04,  1.09s/it]

Epoch 35, Loss: 0.0016


100%|██████████| 40/40 [00:53<00:00,  1.33s/it]


In [15]:
model.eval()
with torch.no_grad():
    predictions = model(torch.tensor(X_test, dtype=torch.float32))
    test_loss = criterion(predictions, torch.tensor(y_test, dtype=torch.float32))

print("Test MSE:", test_loss.item())

Test MSE: 0.0015114840352907777


In [16]:
def predict_next_aqi(model, recent_aqi_values, scaler, seq_len):

    if len(recent_aqi_values) != seq_len:
        raise ValueError(f"Please provide exactly {seq_len} AQI values")

    # Convert to numpy and scale
    recent_aqi_values = np.array(recent_aqi_values).reshape(-1, 1)
    recent_aqi_scaled = scaler.transform(recent_aqi_values)

    # Create tensor [1, seq_len, 1]
    input_tensor = torch.tensor(
        recent_aqi_scaled.reshape(1, seq_len, 1),
        dtype=torch.float32
    )

    # Prediction
    model.eval()
    with torch.no_grad():
        pred_scaled = model(input_tensor).numpy()

    # Inverse scale
    predicted_aqi = scaler.inverse_transform(pred_scaled)[0][0]

    return predicted_aqi

In [17]:
recent_aqi = [55, 60, 58, 62, 65, 63, 66]  # last 7 days AQI
predicted_aqi = predict_next_aqi(
    model=model,
    recent_aqi_values=recent_aqi,
    scaler=scaler,
    seq_len=7
)

print(f"Predicted AQI for next day: {predicted_aqi:.2f}")


Predicted AQI for next day: 56.13


c:\Users\Zubayer Ishfar Zeem\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [18]:
def aqi_status(aqi):
    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Moderate"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <= 200:
        return "Unhealthy"
    elif aqi <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"
print("AQI Status:", aqi_status(predicted_aqi))

AQI Status: Moderate


In [19]:
import os
import torch
from IPython.display import FileLink, display

SAVE_PATH = "aqi_lstm_checkpoint.pth"

checkpoint = {
    # Trained model weights
    "model_state_dict": {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    },

    # Sequence length used during training
    "seq_len": SEQ_LEN,

    # MinMaxScaler parameters
    # Needed to scale user input exactly like training data
    "scaler_scale": float(scaler.scale_[0]),
    "scaler_min": float(scaler.min_[0]),

    # Extra model information
    "input_size": 1,
    "hidden_size": 64,
    "num_layers": 2,
}

torch.save(checkpoint, SAVE_PATH)

print("✅ Model saved successfully!")
print("Path:", SAVE_PATH)
print(
    "File size:",
    round(os.path.getsize(SAVE_PATH) / (1024 * 1024), 2),
    "MB"
)


✅ Model saved successfully!
Path: aqi_lstm_checkpoint.pth
File size: 0.2 MB
